# BT-UNet Evaluation Pipeline
Loads saved models and evaluates them against train, validation, and test splits using the same pipeline as the training notebooks.

## Configuration
Edit `MODELS_TO_EVALUATE` and `MODEL_DIR` to match your saved model filenames and location.

In [15]:
# ============================================================
# CONFIGURATION — edit these to match your saved models
# ============================================================


MODELS_TO_EVALUATE = [
    "barlow_twins_unet.keras",
    "model_BT-Unet_5.keras",
    "model_BT-Unet_50.keras",
    "model_Unet-without-BT-50.keras",
    "model_Unet-without-BT.keras",
]

MODEL_DIR     = "saved_models"   # folder where .keras files live
THRESHOLD     = 0.5              # binarisation threshold
TRAIN_PATH = "datasets/KDSB/train/"
TEST_PATH  = "datasets/KDSB/test/"
IMG_HEIGHT    = 256
IMG_WIDTH     = 256
IMG_CHANNELS  = 3
val_split     = 0.3              # must match the split used during training

## Imports

In [8]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

from skimage.io import imread
from skimage.transform import resize
from skimage.morphology import label

import tensorflow as tf
from tensorflow.keras.metrics import Accuracy, Precision, Recall, MeanIoU, MeanAbsoluteError
from tensorflow.keras import backend as K
from tensorflow.keras.models import load_model
from hausdorff import hausdorff_distance

## Data Loading
Identical to the training notebooks.

In [9]:
train_ids = next(os.walk(TRAIN_PATH + 'org/'))[2][:]
test_ids  = next(os.walk(TEST_PATH  + 'org/'))[2][:]
print(f"Train images: {len(train_ids)}, Test images: {len(test_ids)}")

X_train = np.zeros((len(train_ids), IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS), dtype=np.float32)
Y_train = np.zeros((len(train_ids), IMG_HEIGHT, IMG_WIDTH, 1),            dtype=np.float32)
X_test  = np.zeros((len(test_ids),  IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS), dtype=np.float32)
Y_test  = np.zeros((len(test_ids),  IMG_HEIGHT, IMG_WIDTH, 1),            dtype=np.float32)

print('Loading and resizing test images and masks...')
for n, id_ in tqdm(enumerate(train_ids), total=len(train_ids)):
    img  = imread(TRAIN_PATH + 'org/' + id_)[:, :, :IMG_CHANNELS]
    img  = resize(img, (IMG_HEIGHT, IMG_WIDTH), mode='constant', preserve_range=True)
    X_train[n] = img
    mask_path  = TRAIN_PATH + 'gt/' + id_[:-4] + '_GT.png'
    mask = imread(mask_path)
    mask = np.expand_dims(mask, axis=-1)
    mask = resize(mask, (IMG_WIDTH, IMG_HEIGHT, 1), mode='constant', preserve_range=True)
    mask = mask / 255
    Y_train[n][mask[:, :, 0] > 0.] = 1.

for n, id_ in tqdm(enumerate(test_ids), total=len(test_ids)):
    img  = imread(TEST_PATH + 'org/' + id_)[:, :, :IMG_CHANNELS]
    img  = resize(img, (IMG_HEIGHT, IMG_WIDTH), mode='constant', preserve_range=True)
    X_test[n] = img
    mask_path  = TEST_PATH + 'gt/' + id_[:-4] + '_GT.png'
    mask = imread(mask_path)
    mask = np.expand_dims(mask, axis=-1)
    mask = resize(mask, (IMG_WIDTH, IMG_HEIGHT, 1), mode='constant', preserve_range=True)
    mask = mask / 255
    Y_test[n][mask[:, :, 0] > 0.] = 1.

# 50 % subset used for training (same as training notebooks)
X_train_20 = X_train[:int(X_train.shape[0] * 0.5)]
Y_train_20 = Y_train[:int(Y_train.shape[0] * 0.5)]

# Recreate the exact train / val split
split_idx    = int(X_train_20.shape[0] * (1 - val_split))
X_train_split = X_train_20[:split_idx]
Y_train_split = Y_train_20[:split_idx]
X_val         = X_train_20[split_idx:]
Y_val         = Y_train_20[split_idx:]

print(f"X_train_split: {X_train_split.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}")

np.save("KDSB_X_train_256x256.npy", X_train)
np.save("KDSB_Y_train_256x256.npy", Y_train)
np.save("KDSB_X_test_256x256.npy",  X_test)
np.save("KDSB_Y_test_256x256.npy",  Y_test)
print(f"Saved: {X_train.shape} train images, {X_test.shape} test images")



Train images: 2594, Test images: 1000
Loading and resizing test images and masks...


100%|██████████| 1000/1000 [14:41<00:00,  1.13it/s] 

X_train_split: (907, 256, 256, 3), X_val: (390, 256, 256, 3), X_test: (1000, 256, 256, 3)


Saved: (2594, 256, 256, 3) train images, (1000, 256, 256, 3) test images


X_train = np.load("KDSB_X_train_256x256.npy")
Y_train = np.load("KDSB_Y_train_256x256.npy")
X_test  = np.load("KDSB_X_test_256x256.npy")
Y_test  = np.load("KDSB_Y_test_256x256.npy")
print(f"Loaded: {X_train.shape} train, {X_test.shape} test")

## Metric Helpers
Identical to the training notebooks.

In [ ]:
def iou_metric(y_true_in, y_pred_in, print_table=False):
    labels = label(y_true_in > 0.5)
    y_pred = label(y_pred_in > 0.5)
    true_objects = len(np.unique(labels))
    pred_objects = len(np.unique(y_pred))
    intersection = np.histogram2d(labels.flatten(), y_pred.flatten(),
                                  bins=(true_objects, pred_objects))[0]
    area_true = np.expand_dims(np.histogram(labels, bins=true_objects)[0], -1)
    area_pred = np.expand_dims(np.histogram(y_pred, bins=pred_objects)[0],  0)
    union = area_true + area_pred - intersection
    intersection = intersection[1:, 1:]
    union = union[1:, 1:]
    union[union == 0] = 1e-9
    iou = intersection / union

    def precision_at(threshold, iou):
        matches = iou > threshold
        tp = np.sum(np.sum(matches, axis=1) == 1)
        fp = np.sum(np.sum(matches, axis=0) == 0)
        fn = np.sum(np.sum(matches, axis=1) == 0)
        return tp, fp, fn

    prec = []
    for t in np.arange(0.5, 1.0, 0.05):
        tp, fp, fn = precision_at(t, iou)
        p = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0
        prec.append(p)
    return np.mean(prec)


def iou_metric_batch(y_true_in, y_pred_in):
    batch_size = y_true_in.shape[0]
    value = 0.
    for batch in range(batch_size):
        value += iou_metric(y_true_in[batch], y_pred_in[batch])
    return value / batch_size


def dice_coeff(y_true, y_pred):
    smooth = 1.
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)


def haud_dist(y_true, y_pred):
    return hausdorff_distance(np.squeeze(y_true), np.squeeze(y_pred))


def haud_dist_batch(y_true, y_pred):
    if len(y_true.shape) == 2:
        return haud_dist(y_true, y_pred)
    hd = 0.
    for batch in range(y_true.shape[0]):
        hd += haud_dist(y_true[batch], y_pred[batch])
    return hd / y_true.shape[0]


def evalResult(gt, pred, num_class=2):
    """Identical computation to training notebooks; also returns a metrics dict."""
    gt   = np.squeeze(gt)
    pred = np.squeeze(pred)

    acc = Accuracy()
    acc.update_state(gt, pred)
    r_acc = acc.result().numpy()

    pr = Precision()
    pr.update_state(gt, pred)
    r_pr = pr.result().numpy()

    rc = Recall()
    rc.update_state(gt, pred)
    r_rc = rc.result().numpy()

    mi = MeanIoU(num_class)
    mi.update_state(gt, pred)
    r_mi = mi.result().numpy()

    dc = 0.
    for img in range(gt.shape[0]):
        dc += dice_coeff(gt[img], pred[img]).numpy()
    dc /= gt.shape[0]

    hd   = haud_dist_batch(gt, pred)
    miou = iou_metric_batch(gt, pred)

    mae = MeanAbsoluteError()
    r_mae = mae(gt, pred).numpy()

    print(f"  Accuracy={r_acc:.4f}  Precision={r_pr:.4f}  Recall={r_rc:.4f}  "
          f"MeanIoU={r_mi:.4f}  Dice={dc:.4f}  HD={hd:.4f}  MyIoU={miou:.4f}  MAE={r_mae:.4f}")

    return {
        "Accuracy":  r_acc,
        "Precision": r_pr,
        "Recall":    r_rc,
        "MeanIoU":   r_mi,
        "Dice":      dc,
        "HD":        hd,
        "MyIoU":     miou,
        "MAE":       r_mae,
    }

## Evaluation Loop

In [16]:
results = []

for model_name in MODELS_TO_EVALUATE:

    model_path = os.path.join(MODEL_DIR, model_name)

    if not os.path.exists(model_path):
        print(f"\n[SKIP] Model not found: {model_path}")
        continue

    print(f"\n{'='*60}")
    print(f"Evaluating: {model_name}")
    print(f"{'='*60}")

    model = load_model(model_path, compile=False)

    # ---- Predictions (same pipeline as training notebooks) ----
    preds_train = model.predict(X_train_split, verbose=1)
    preds_x     = model.predict(X_train,       verbose=1)
    preds_val   = model.predict(X_val,          verbose=1)
    preds_test  = model.predict(X_test,         verbose=1)

    # ---- Threshold ----
    preds_train_t = (preds_train > THRESHOLD).astype(np.float32)
    preds_x_t     = (preds_x     > THRESHOLD).astype(np.float32)
    preds_val_t   = (preds_val   > THRESHOLD).astype(np.float32)
    preds_test_t  = (preds_test  > THRESHOLD).astype(np.float32)

    # ---- Evaluate ----
    print("\nTRAIN")
    train_m = evalResult(Y_train_split.astype(np.float32), preds_train_t)

    print("\nFULL DATASET")
    full_m  = evalResult(Y_train.astype(np.float32),       preds_x_t)

    print("\nVALIDATION")
    val_m   = evalResult(Y_val.astype(np.float32),         preds_val_t)

    print("\nTEST")
    test_m  = evalResult(Y_test.astype(np.float32),        preds_test_t)

    # ---- Collect ----
    row = {"Model": model_name}
    for split_name, metrics in [("Train", train_m), ("Full", full_m), ("Val", val_m), ("Test", test_m)]:
        for k, v in metrics.items():
            row[f"{split_name} {k}"] = v
    results.append(row)

print("\nDone.")


Evaluating: barlow_twins_unet.keras
29/29 ━━━━━━━━━━━━━━━━━━━━ 5s 136ms/step
82/82 ━━━━━━━━━━━━━━━━━━━━ 10s 127ms/step
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 119ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 4s 138ms/step

TRAIN


NameError: name 'evalResult' is not defined

In [12]:
import os
print(os.getcwd())
print(os.listdir("."))

c:\Users\ujai264\Self-Supervised-Barlow-Twins-Pretraining-for-Data-Efficient-U-Net-Medical-Image-Segmentation\BT Unet
['barlow_twins_unet.keras', 'BT-Unet.ipynb', 'BT-Unet.py', 'BT_Evaluation.ipynb', 'datasets', 'evaluation_results.csv', 'KDSB_X_test_256x256.npy', 'KDSB_X_train_256x256.npy', 'KDSB_Y_test_256x256.npy', 'KDSB_Y_train_256x256.npy', 'logs', 'model_BT-Unet_5.keras', 'model_BT-Unet_50.keras', 'model_Unet-without-BT-50.keras', 'model_Unet-without-BT.keras', 'README.md', 'Unet-without-BT.ipynb']


## Results Table

In [ ]:
results_df = pd.DataFrame(results)
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)

print("\n================ FINAL RESULTS ================\n")
display(results_df)

results_df.to_csv("evaluation_results.csv", index=False)
print("\nSaved to evaluation_results.csv")

## Per-metric Summary (Test set)

In [ ]:
if not results_df.empty:
    test_cols = [c for c in results_df.columns if c.startswith("Test")]
    display(results_df[["Model"] + test_cols])